# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-6, 6, 1000)
y = np.sinc(x)

plt.plot(x, y, color="darkblue")
plt.axhline(0, color="black", linewidth=0.5)
plt.axvline(0, color="black", linewidth=0.5)
plt.grid(True, alpha=0.3)

In [ ]:
!nvcc --version
%uv pip install datasets
%uv pip install transformers

In [ ]:
# ── HF cache redirect MUST precede transformers/datasets import ─────────────
import os
os.environ["HF_HOME"] = "/mnt/hf-cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/hf-cache/datasets"

import json
import queue
import random
import threading

import numpy as np
import torch
from datasets import load_dataset
from safetensors.torch import save_file
from tqdm import tqdm
from transformers import AutoModel, AutoTokenizer

# ── Configuration ───────────────────────────────────────────────────────────
SEED = 42
TARGET_TOKENS = 5_500_000
LAYERS = [1, 12, 20, 24]
LAST_LAYER = max(LAYERS)
OUTPUT_BASE = "/mnt/fineweb-acts"          # ← persistent Volume mount
MODEL_NAME = "google/gemma-2-2b"
DATASET_NAME = "HuggingFaceFW/fineweb"
DATASET_CONFIG = "CC-MAIN-2024-10"
MAX_SEQ_LEN = 4096
STORE_DTYPE = torch.bfloat16

MAX_TOKENS_PER_BATCH = 32 * 1024
SORT_WINDOW = 8192
PREFETCH_DEPTH = 64
RAW_DEPTH = 4096
N_TOKENIZE_WORKERS = 6
SHARD_TOKENS = 250_000                       # ← smaller: earlier flush, bounded RAM

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

for layer in LAYERS:
    os.makedirs(f"{OUTPUT_BASE}/layer_{layer}", exist_ok=True)
os.makedirs(f"{OUTPUT_BASE}/text-samples", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Optional explicit Volume commit handle; falls back to Modal's automatic
# background commits if the handle cannot be resolved inside the notebook.
try:
    import modal
    _acts_vol = modal.Volume.from_name("fineweb-acts")
except Exception:
    _acts_vol = None


def _commit() -> None:
    if _acts_vol is not None:
        try:
            _acts_vol.commit()
        except Exception as e:
            tqdm.write(f"volume commit skipped: {e}")


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

try:
    model = AutoModel.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )
except Exception:
    model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)

model.eval().to(device)
torch.set_grad_enabled(False)

# ── Hooks with early-exit after the deepest target layer ────────────────────
activations: dict[int, torch.Tensor] = {}


class _StopForward(Exception):
    pass


def make_hook(layer_idx: int):
    def hook(_module, _input, output):
        hidden = output[0] if isinstance(output, tuple) else output
        activations[layer_idx] = hidden.detach()
        if layer_idx == LAST_LAYER:
            raise _StopForward
    return hook


hooks = [model.layers[i].register_forward_hook(make_hook(i)) for i in LAYERS]

# ── Pipeline plumbing ───────────────────────────────────────────────────────
_SENTINEL = object()
stop_event = threading.Event()

raw_q: queue.Queue = queue.Queue(maxsize=RAW_DEPTH)
tok_q: queue.Queue = queue.Queue(maxsize=RAW_DEPTH)
batch_q: queue.Queue = queue.Queue(maxsize=PREFETCH_DEPTH)
writer_q: queue.Queue = queue.Queue(maxsize=8)


def _put(q, item) -> bool:
    while True:
        try:
            q.put(item, timeout=0.5)
            return True
        except queue.Full:
            if stop_event.is_set():
                return False


def reader() -> None:
    ds = load_dataset(DATASET_NAME, name=DATASET_CONFIG, split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=10_000)
    for example in ds:
        if stop_event.is_set():
            break
        text = example.get("text", "")
        if text.strip() and not _put(raw_q, text):
            break
    for _ in range(N_TOKENIZE_WORKERS):
        _put(raw_q, _SENTINEL)


def tokenize_worker() -> None:
    while not stop_event.is_set():
        try:
            item = raw_q.get(timeout=0.5)
        except queue.Empty:
            continue
        if item is _SENTINEL:
            _put(tok_q, _SENTINEL)
            return
        ids = tokenizer(item, truncation=True, max_length=MAX_SEQ_LEN)["input_ids"]
        if ids:
            _put(tok_q, (item, ids))


def emit_batch(batch) -> None:
    texts = [b[0] for b in batch]
    seqs = [b[1] for b in batch]
    max_len = max(len(s) for s in seqs)
    pad_id = tokenizer.pad_token_id
    input_ids = torch.full((len(seqs), max_len), pad_id, dtype=torch.long)
    attn = torch.zeros((len(seqs), max_len), dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, : len(s)] = torch.tensor(s, dtype=torch.long)
        attn[i, : len(s)] = 1
    _put(batch_q, (texts, input_ids.pin_memory(), attn.pin_memory()))


def batcher() -> None:
    window = []
    done = 0

    def flush_window() -> None:
        if not window:
            return
        window.sort(key=lambda x: len(x[1]))
        batch, cur_max = [], 0
        for text, ids in window:
            cand_max = max(cur_max, len(ids))
            if batch and (len(batch) + 1) * cand_max > MAX_TOKENS_PER_BATCH:
                emit_batch(batch)
                batch, cur_max = [], len(ids)
            else:
                cur_max = cand_max
            batch.append((text, ids))
        if batch:
            emit_batch(batch)
        window.clear()

    while True:
        try:
            item = tok_q.get(timeout=0.5)
        except queue.Empty:
            if stop_event.is_set():
                return
            continue
        if item is _SENTINEL:
            done += 1
            if done == N_TOKENIZE_WORKERS:
                flush_window()
                _put(batch_q, _SENTINEL)
                return
            continue
        window.append(item)
        if len(window) >= SORT_WINDOW:
            flush_window()


def writer() -> None:
    buffers = {l: [] for l in LAYERS}
    meta_f = open(f"{OUTPUT_BASE}/metadata.jsonl", "w", encoding="utf-8")
    state = {"tokens": 0, "shard": 0, "offset": 0, "row": 0, "sample": 0, "text_f": None}

    def text_file():
        if state["text_f"] is None:
            state["text_f"] = open(
                f"{OUTPUT_BASE}/text-samples/shard_{state['shard']}.jsonl",
                "w", encoding="utf-8",
            )
        return state["text_f"]

    def flush() -> None:
        if state["tokens"] == 0:
            return
        for l in LAYERS:
            big = torch.cat(buffers[l], dim=0).contiguous()
            save_file({"activations": big},
                      f"{OUTPUT_BASE}/layer_{l}/shard_{state['shard']}.safetensors")
            buffers[l].clear()
        if state["text_f"] is not None:
            state["text_f"].close()
            state["text_f"] = None
        meta_f.flush()
        _commit()                                       # ← persist shard durably
        state["shard"] += 1
        state["tokens"] = state["offset"] = state["row"] = 0

    while True:
        item = writer_q.get()
        if item is _SENTINEL:
            break
        host_layers, lengths, texts = item
        for l in LAYERS:
            buffers[l].append(host_layers[l])
        tf = text_file()
        for length, txt in zip(lengths, texts):
            meta_f.write(json.dumps({
                "sample": state["sample"], "shard": state["shard"],
                "offset": state["offset"], "length": length, "row": state["row"],
            }) + "\n")
            tf.write(json.dumps({"text": txt}) + "\n")
            state["offset"] += length
            state["row"] += 1
            state["sample"] += 1
        state["tokens"] += sum(lengths)
        if state["tokens"] >= SHARD_TOKENS:
            flush()

    flush()
    meta_f.close()
    _commit()                                           # ← final trailing shard


# ── Launch producers and writer ─────────────────────────────────────────────
threading.Thread(target=reader, daemon=True).start()
for _ in range(N_TOKENIZE_WORKERS):
    threading.Thread(target=tokenize_worker, daemon=True).start()
threading.Thread(target=batcher, daemon=True).start()

writer_thread = threading.Thread(target=writer)         # non-daemon: must flush
writer_thread.start()

# ── Main GPU loop ───────────────────────────────────────────────────────────
total_tokens = 0
pbar = tqdm(total=TARGET_TOKENS, unit="tok", desc="collecting")

while total_tokens < TARGET_TOKENS:
    try:
        item = batch_q.get(timeout=1.0)
    except queue.Empty:
        continue
    if item is _SENTINEL:
        break

    texts, input_ids, attn = item
    lengths = attn.sum(dim=1).tolist()
    input_ids = input_ids.to(device, non_blocking=True)
    attn = attn.to(device, non_blocking=True)

    activations.clear()
    try:
        model(input_ids=input_ids, attention_mask=attn, use_cache=False)
    except _StopForward:
        pass
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        tqdm.write(f"OOM (shape={list(input_ids.shape)}), skipping")
        continue

    host_layers = {}
    for layer in LAYERS:
        h = activations[layer]
        packed = torch.cat([h[b, :int(L)] for b, L in enumerate(lengths)], dim=0)
        host = torch.empty(tuple(packed.shape), dtype=STORE_DTYPE, pin_memory=True)
        host.copy_(packed, non_blocking=True)
        host_layers[layer] = host
    torch.cuda.synchronize()

    writer_q.put((host_layers, [int(L) for L in lengths], texts))

    batch_tokens = int(sum(lengths))
    total_tokens += batch_tokens
    pbar.update(batch_tokens)

# ── Teardown ────────────────────────────────────────────────────────────────
stop_event.set()
writer_q.put(_SENTINEL)
writer_thread.join()
pbar.close()
for h in hooks:
    h.remove()
print(f"Done. Tokens: {total_tokens:,}")

# SVD GENERATOR BELOW
- Checks GPU capacity stores and saves it

In [ ]:
"""
svd_capacity.py — capacity under torch.linalg.svd(full_matrices=False).

Peak ≈ k · M · N + M²  (float32 elements), with k ≈ 3:
  input A + internal copy + output U, each N·M; Vh and scratch are O(M²).
Solving for N:
      N ≤ (B·safety / bpe − M²) / (k · M)
"""

import torch

try:
    import psutil
except ImportError:
    psutil = None


def available_bytes(device: str = "cpu") -> int:
    if device == "cuda" and torch.cuda.is_available():
        free, _ = torch.cuda.mem_get_info()
        return free
    if psutil is not None:
        return psutil.virtual_memory().available
    raise RuntimeError("Install psutil or supply the budget explicitly.")


def max_samples(
    budget_bytes: int,
    *,
    dim: int = 2304,          # M = D; also the effective rank r of the thin SVD
    bytes_per_elt: int = 4,   # float32 mandated by linalg.svd
    overhead: float = 3.0,    # k: input + working copy + U
    safety: float = 0.85,     # headroom vs fragmentation / peak transients
) -> int:
    """Largest N admissible to a dense thin SVD within budget_bytes."""
    M = dim
    elt_budget = (budget_bytes * safety) / bytes_per_elt
    numerator = elt_budget - (M * M)        # subtract the O(M²) Vh + scratch
    denominator = overhead * M              # the k · N · M regime
    return max(0, int(numerator // denominator))


def report(device: str = "cpu", dim: int = 2304) -> None:
    free = available_bytes(device)
    n = max_samples(free, dim=dim)
    print(f"device={device}  free={free / 2**30:6.2f} GiB  M=D={dim}  k=3")
    print(f"max N (rows admissible to thin SVD): {n:,}")
    print(f"  data matrix at that N: {(dim * n * 4) / 2**30:.2f} GiB")
    print(f"  projected peak (k·N·M·4): {(3 * dim * n * 4) / 2**30:.2f} GiB")
    # Sensitivity to the overhead constant, since k dominates the estimate.
    for k in (2.0, 2.5, 3.0):
        nk = max_samples(free, dim=dim, overhead=k)
        print(f"    k={k}: N ≤ {nk:,}")


if __name__ == "__main__":
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    report(device=dev)

# Tall-Thin SVD Calculation

In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  svd_run.py — collate one layer, audit integrity, thin-SVD, persist      ║
# ║                                                                          ║
# ║  Notebook-ready. Configure CONFIG, then run top to bottom.               ║
# ║                                                                          ║
# ║  Pipeline:                                                               ║
# ║    1. collate every shard of LAYER into one (N, M) float32 matrix        ║
# ║    2. audit finiteness per-row; filter or raise per NONFINITE_POLICY     ║
# ║    3. guard against the dense thin-SVD memory ceiling before committing  ║
# ║    4. torch.linalg.svd(full_matrices=False)                              ║
# ║    5. save U / S_Vh / kept_indices / manifest to OUTPUT_BASE/svd/        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os
import re
import json
import time

import torch
from safetensors.torch import load_file, save_file

try:
    import psutil
except ImportError:
    psutil = None





# ── Capacity guard ───────────────────────────────────────────────────────────
def available_bytes(device: str) -> int:
    """Free memory on the target device, in bytes."""
    if device == "cuda" and torch.cuda.is_available():
        free, _ = torch.cuda.mem_get_info()
        return free
    if psutil is not None:
        return psutil.virtual_memory().available
    raise RuntimeError("Install psutil or set the budget manually.")


def max_admissible_N(budget: int, dim: int) -> int:
    """Largest N a dense thin SVD tolerates: N ≤ (B·safety/bpe − M²)/(k·M)."""
    elt = (budget * SAFETY) / BYTES_PER_ELT
    return max(0, int((elt - dim * dim) // (OVERHEAD * dim)))


# ── Collation ────────────────────────────────────────────────────────────────
def _shard_no(fn: str) -> int:
    """Numeric shard ordinal, so sorting is numeric rather than lexical."""
    return int(re.search(r"shard_(\d+)", fn).group(1))


def collate_layer(base: str, layer: int, dtype: torch.dtype) -> torch.Tensor:
    """Vertically concatenate all shards of `layer` into one (N, M) tensor."""
    layer_dir = os.path.join(base, f"layer_{layer}")
    files = sorted(
        (f for f in os.listdir(layer_dir) if f.endswith(".safetensors")),
        key=_shard_no,
    )
    if not files:
        raise FileNotFoundError(f"No shards in {layer_dir}")
    chunks = [load_file(os.path.join(layer_dir, f))["activations"].to(dtype)
              for f in files]
    return torch.cat(chunks, dim=0)


# ── Integrity audit ──────────────────────────────────────────────────────────
def check_tensor_integrity(A: torch.Tensor):
    """
    Diagnose non-finite contamination of the collated matrix, per-row, so a
    localised defect costs only its tokens rather than the whole decomposition.

    Returns (report, row_ok) where row_ok is a boolean (N,) mask, True ⇒ finite.
    """
    finite = torch.isfinite(A)
    row_ok = finite.all(dim=1)
    bad_rows = torch.nonzero(~row_ok, as_tuple=False).flatten()

    report = {
        "rows": A.shape[0],
        "cols": A.shape[1],
        "nan_elements": int(torch.isnan(A).sum()),
        "posinf_elements": int(torch.isposinf(A).sum()),
        "neginf_elements": int(torch.isneginf(A).sum()),
        "bad_rows": int(bad_rows.numel()),
        "bad_row_indices": bad_rows[:64].tolist(),     # first 64, for inspection
        "clean": bool(row_ok.all()),
        "abs_max_finite": float(A[finite].abs().max()) if finite.any() else float("nan"),
    }
    return report, row_ok


def sanitize(A: torch.Tensor, row_ok: torch.Tensor, policy: str):
    """
    Apply NONFINITE_POLICY. On "filter" returns (A_clean, kept_indices) mapping
    survivors back to original global indices — the correspondence the reverse
    index (ActivationIndex) presupposes. On "raise", aborts unless clean.
    """
    if bool(row_ok.all()):
        return A, None
    if policy == "raise":
        raise ValueError(
            "Non-finite rows present; set NONFINITE_POLICY='filter' to drop "
            "them, or inspect the offending samples before proceeding."
        )
    kept = torch.nonzero(row_ok, as_tuple=False).flatten()
    return A[kept].contiguous(), kept


# ── CONFIG (addition) ────────────────────────────────────────────────────────
SVD_METHOD   = "gram"     # "gram" → Gram-matrix route (robust for tall N≫M);
                          # "direct" → torch.linalg.svd (the path that just failed)
EPS_FLOOR    = 1e-12      # singular values below σ_max·EPS_FLOOR are treated as 0


def svd_tall(A: torch.Tensor, method: str = "gram",
             eps_floor: float = 1e-12):
    """
    Thin SVD robust to cuSOLVER's gesvdj size ceiling on very tall matrices.

    method="gram": eigendecompose the M×M Gram matrix G = AᵀA (a 2304² problem),
        yielding V and σ² directly; recover U = A·V·Σ⁻¹. Avoids the tall-matrix
        driver path altogether. Numerical caveat: the Gram product squares the
        condition number, so singular values below ≈ √eps · σ_max lose precision
        — immaterial for PCA / interpretability of the dominant directions, but
        unsuitable for rank-revealing a near-singular matrix.
    method="direct": the original torch.linalg.svd(full_matrices=False).

    Returns (U, S, Vh) with the same shapes and descending-σ convention as
    torch.linalg.svd, so the surrounding save/manifest code is unchanged.
    """
    if method == "direct":
        return torch.linalg.svd(A, full_matrices=False)

    # Gram route — A:(N,M), N ≫ M
    G = A.transpose(0, 1) @ A                       # (M, M): the only reduction over N
    evals, V = torch.linalg.eigh(G)                 # ascending eigenpairs of a SPD matrix
    evals = evals.flip(0)                           # → descending
    V = V.flip(1)
    S = torch.sqrt(evals.clamp_min(0.0))            # σ = √λ  (λ≥0 up to round-off)

    # U = A V Σ⁻¹, with tiny/zero σ guarded so degenerate columns become zero
    safe = S > (S.max() * eps_floor)
    Sinv = torch.where(safe, 1.0 / S, torch.zeros_like(S))
    U = (A @ V) * Sinv.unsqueeze(0)                 # (N, M)
    Vh = V.transpose(0, 1)                          # (M, M)
    return U, S, Vh

    


In [4]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_BASE      = "/mnt/fineweb-acts"   # dataset run produced by collection
LAYER            = 24                      # which layer's activations to decompose

SVD_SUBDIR       = "svd"                   # new folder created beneath OUTPUT_BASE
DIM              = 2304                    # M = D (Gemma-2-2b hidden size)
KEEP_RANK        = None                    # post-hoc truncation; None ⇒ retain all M
COMPUTE_DTYPE    = torch.float32           # mandatory: linalg.svd rejects bf16
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"

# integrity / backend policy
NONFINITE_POLICY = "filter"               # "filter" → drop bad rows; "raise" → abort
PREFERRED_LINALG = None                   # e.g. "magma" to sidestep cuSOLVER; None ⇒ default

# memory-model constants (dense thin-SVD: peak ≈ k·N·M + M²)
OVERHEAD         = 3.0                     # k: input + working copy + U, each N·M
SAFETY           = 0.85                    # headroom vs fragmentation / peaks
BYTES_PER_ELT    = 4                       # float32

In [5]:
base, layer = OUTPUT_BASE, LAYER

out_dir = os.path.join(base, SVD_SUBDIR, f"layer_{layer}")
os.makedirs(out_dir, exist_ok=True)

if PREFERRED_LINALG is not None:
    torch.backends.cuda.preferred_linalg_library(PREFERRED_LINALG)

# 1 ── collate ────────────────────────────────────────────────────────────
print(f"[1/5] collating layer {layer} …")
A = collate_layer(base, layer, COMPUTE_DTYPE)
N0, M = A.shape
assert M == DIM, f"expected M={DIM}, got {M}"
print(f"      collated: N={N0:,}  M={M}  "
      f"({A.numel() * BYTES_PER_ELT / 2**30:.2f} GiB)")

# 2 ── audit integrity ──────────────────────────────────────────────────��─
print(f"[2/5] auditing integrity …")
report, row_ok = check_tensor_integrity(A)
print(f"      nan={report['nan_elements']:,}  +inf={report['posinf_elements']:,}  "
      f"-inf={report['neginf_elements']:,}  bad_rows={report['bad_rows']:,}  "
      f"|max|={report['abs_max_finite']:.3g}")
if not report["clean"]:
    print(f"      offending rows (first few): {report['bad_row_indices']}")
A, kept_indices = sanitize(A, row_ok, NONFINITE_POLICY)
N = A.shape[0]
if kept_indices is not None:
    print(f"      filtered {N0 - N:,} non-finite rows → N={N:,}")

# 3 ── capacity guard (before device transfer / decomposition) ────────────
budget = available_bytes(DEVICE)
cap = max_admissible_N(budget, M)
print(f"[3/5] budget={budget / 2**30:.2f} GiB on {DEVICE}  →  N_max≈{cap:,}")
if N > cap:
    raise MemoryError(
        f"N={N:,} exceeds the dense thin-SVD ceiling N_max≈{cap:,} (k={OVERHEAD}). "
        f"Use torch.svd_lowrank(A, q=rank) for a rank-bounded working set, "
        f"or decompose on a larger-memory device."
    )

# 4 ── SVD ───────────────────────────────────────────────────────────
print(f"[4/5] SVD on {DEVICE} (method={SVD_METHOD}) …")
t0 = time.time()
A = A.to(DEVICE)
U, S, Vh = svd_tall(A, method=SVD_METHOD, eps_floor=EPS_FLOOR)
if DEVICE == "cuda":
    torch.cuda.synchronize()
elapsed = time.time() - t0
if KEEP_RANK is not None:
    U, S, Vh = U[:, :KEEP_RANK], S[:KEEP_RANK], Vh[:KEEP_RANK, :]
k = S.shape[0]
print(f"      done in {elapsed:.1f}s  →  U:{tuple(U.shape)} "
      f"S:{tuple(S.shape)} Vh:{tuple(Vh.shape)}")

# 5 ── persist ────────────────────────────────────────────────────────────
print(f"[5/5] saving to {out_dir} …")
# U carries the entire N·M mass; segregate it so S/Vh load cheaply alone.
save_file({"U": U.cpu().contiguous()},  os.path.join(out_dir, "U.safetensors"))
save_file({"S": S.cpu().contiguous(),
           "Vh": Vh.cpu().contiguous()}, os.path.join(out_dir, "S_Vh.safetensors"))
# Survivor map: load-bearing when filtering, since excision renumbers rows.
if kept_indices is not None:
    save_file({"kept_indices": kept_indices.to(torch.long)},
              os.path.join(out_dir, "kept_indices.safetensors"))

manifest = {
    "layer": layer, "N_original": N0, "N_decomposed": N, "M": M, "rank": k,
    "full_matrices": False, "keep_rank": KEEP_RANK,
    "compute_dtype": str(COMPUTE_DTYPE), "device": DEVICE,
    "preferred_linalg": PREFERRED_LINALG,
    "nonfinite_policy": NONFINITE_POLICY, "rows_filtered": N0 - N,
    "integrity": {kk: report[kk] for kk in
                  ("nan_elements", "posinf_elements", "neginf_elements",
                   "bad_rows", "abs_max_finite")},
    "elapsed_seconds": round(elapsed, 2),
    "total_variance": float((S ** 2).sum().cpu()),
    "source": os.path.join(base, f"layer_{layer}"),
}
with open(os.path.join(out_dir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

# If OUTPUT_BASE is a Modal Volume, make the new shard durable.
try:
    import modal
    modal.Volume.from_name(os.path.basename(base.rstrip("/"))).commit()
except Exception:
    pass

print(f"✓ saved U / S_Vh / manifest under {out_dir}"
      + (f"  (kept_indices.safetensors records the {N:,} survivors)"
         if kept_indices is not None else ""))

[1/5] collating layer 24 …
      collated: N=5,532,685  M=2304  (47.49 GiB)
[2/5] auditing integrity …
      nan=0  +inf=0  -inf=0  bad_rows=0  |max|=2.9e+03
[3/5] budget=177.74 GiB on cuda  →  N_max≈5,866,564
[4/5] SVD on cuda (method=gram) …
      done in 4.4s  →  U:(5532685, 2304) S:(2304,) Vh:(2304, 2304)
[5/5] saving to /mnt/fineweb-acts/svd/layer_24 …
✓ saved U / S_Vh / manifest under /mnt/fineweb-acts/svd/layer_24


In [ ]:
# ── Verification ─────────────────────────────────────────────────────────────
def verify_svd(out_dir: str, base: str, layer: int, *,
               sample_rows: int = 50_000, tol: float = 1e-3) -> dict:
    """
    Certify the saved factorisation by its defining identities, evaluated on a
    random row-subsample (the full U is gigabyte-scale; a subsample bounds cost
    while remaining statistically decisive).

    Checks, each reported as a relative error against the appropriate norm:
      • reconstruction:  ‖A_s − U_s Σ Vh‖_F / ‖A_s‖_F
      • V orthonormality: ‖VhᵀVh − I‖_F / √M
      • U orthonormality: ‖U_sᵀU_s − (its diagonal)‖ on the subsample
      • spectrum:         σ non-negative and monotonically non-increasing
    A_s is reconstructed from the original shards for the sampled rows, so the
    test is independent of the in-memory matrix that produced the factors.
    """
    U  = load_file(os.path.join(out_dir, "U.safetensors"))["U"]
    sv = load_file(os.path.join(out_dir, "S_Vh.safetensors"))
    S, Vh = sv["S"], sv["Vh"]

    # If rows were filtered, the original matrix must be realigned to survivors.
    kpath = os.path.join(out_dir, "kept_indices.safetensors")
    kept = (load_file(kpath)["kept_indices"]
            if os.path.exists(kpath) else None)

    A_full = collate_layer(base, layer, COMPUTE_DTYPE)
    if kept is not None:
        A_full = A_full[kept]
    N = U.shape[0]
    assert A_full.shape[0] == N, "row count of factors and source disagree"

    g = torch.Generator().manual_seed(0)
    m = min(sample_rows, N)
    rows = torch.randperm(N, generator=g)[:m]

    A_s = A_full[rows].to(COMPUTE_DTYPE)
    U_s = U[rows].to(COMPUTE_DTYPE)
    recon = (U_s * S.unsqueeze(0)) @ Vh                      # (m, M)

    a_norm = torch.linalg.norm(A_s)
    recon_err = float(torch.linalg.norm(A_s - recon) / a_norm)

    M = Vh.shape[1]
    VtV = Vh @ Vh.transpose(0, 1)                            # (M, M) ≈ I
    v_orth_err = float(torch.linalg.norm(VtV - torch.eye(M)) / (M ** 0.5))

    # U columns should be orthogonal; on a subsample we check off-diagonal mass
    # of U_sᵀU_s relative to its diagonal (full U is too large to load whole).
    UtU = U_s.transpose(0, 1) @ U_s
    offdiag = UtU - torch.diag(torch.diag(UtU))
    u_orth_err = float(torch.linalg.norm(offdiag) / torch.linalg.norm(torch.diag(UtU)))

    S_desc = bool(torch.all(S[:-1] - S[1:] >= -tol * S.max()))
    S_nonneg = bool(torch.all(S >= -tol * S.max()))

    # Optional cross-check of the spectrum against the direct driver on a
    # column-narrowed block, small enough to evade the gesvdj size ceiling.
    cross = None
    try:
        blk = A_s[:min(m, 8192)]
        S_direct = torch.linalg.svdvals(blk)
        S_gram = svd_tall(blk, method="gram")[1]
        k = min(S_direct.numel(), S_gram.numel())
        cross = float(torch.linalg.norm(S_direct[:k] - S_gram[:k])
                      / torch.linalg.norm(S_direct[:k]))
    except Exception as e:
        cross = f"skipped ({type(e).__name__})"

    report = {
        "sampled_rows": m,
        "reconstruction_rel_err": recon_err,
        "V_orthonormality_err": v_orth_err,
        "U_orthogonality_err": u_orth_err,
        "spectrum_descending": S_desc,
        "spectrum_nonnegative": S_nonneg,
        "spectrum_cross_check_rel_err": cross,
        "tolerance": tol,
        "passed": (recon_err < tol and v_orth_err < tol
                   and u_orth_err < tol and S_desc and S_nonneg),
    }

    print("\n── SVD verification ─────────────────────────────────────────────")
    print(f"  sampled rows               : {m:,}")
    print(f"  reconstruction rel err     : {recon_err:.2f}   (tol {tol:.0f})")
    print(f"  V orthonormality err       : {v_orth_err:.2f}")
    print(f"  U orthogonality err        : {u_orth_err:.2f}")
    print(f"  spectrum descending        : {S_desc}")
    print(f"  spectrum non-negative      : {S_nonneg}")
    print(f"  spectrum cross-check err   : {cross}")
    print(f"  ⇒ {'PASS' if report['passed'] else 'FAIL'}")
    print("─────────────────────────────────────────────────────────────────")
    return report

out = '/mnt/fineweb-acts/svd/layer_20'
verification = verify_svd(out, OUTPUT_BASE, LAYER,tol=0.1)